# MetroCrowdManager — GRPO Training (Colab T4)

Curriculum:
1. **Phase A** — `ticket_issuance` only. Verifies the agentic loop + reward signal.
2. **Phase B** — `ticket_booking` only. Multi-turn conversation discipline.
3. **Phase C** — mixed (all three tasks). Production run for the demo plots.

Tested on Qwen3-1.7B-Instruct + Unsloth + TRL. ~12 GB VRAM.

Plots are saved to `docs/plots/` so they're committed alongside the notebook.

In [ ]:
# Cell 1 — Setup
!pip install -q --upgrade unsloth trl peft datasets accelerate bitsandbytes openenv fastmcp 2>&1 | tail -3
import os, sys
if not os.path.exists('/content/Gluon'):
    !git clone https://github.com/<YOUR-FORK>/Gluon.git /content/Gluon
%cd /content/Gluon
sys.path.insert(0, '.')
sys.path.insert(0, 'MetroCrowdManager')

In [ ]:
# Cell 2 — Phase A (ticket_issuance, ~1 hr)
!python training/train_grpo.py --phase A --num-episodes 60 --output-dir outputs/phaseA

In [ ]:
# Cell 3 — Phase B (ticket_booking, ~2-3 hr) — resume from Phase A adapters
!python training/train_grpo.py --phase B --num-episodes 60 \
    --resume-adapters outputs/phaseA/final \
    --output-dir outputs/phaseB

In [ ]:
# Cell 4 — Phase C (mixed, ~3-4 hr)
!python training/train_grpo.py --phase C --num-episodes 100 \
    --resume-adapters outputs/phaseB/final \
    --output-dir outputs/phaseC

In [ ]:
# Cell 5 — Plot reward curves (writes PNGs to docs/plots/)
import json, csv, os, matplotlib.pyplot as plt
from collections import defaultdict
os.makedirs('docs/plots', exist_ok=True)
for phase in ('A', 'B', 'C'):
    path = f'outputs/phase{phase}/rewards.csv'
    if not os.path.exists(path):
        continue
    by_task = defaultdict(list)
    steps = defaultdict(list)
    with open(path) as f:
        reader = csv.DictReader(f)
        for row in reader:
            by_task[row['task']].append(float(row['reward']))
            steps[row['task']].append(int(row['step']))
    plt.figure()
    for task, rewards in by_task.items():
        # smooth with rolling mean
        window = max(1, len(rewards) // 20)
        smoothed = [sum(rewards[max(0, i-window):i+1]) / (i - max(0, i-window) + 1) for i in range(len(rewards))]
        plt.plot(steps[task], smoothed, label=task)
    plt.xlabel('Training step')
    plt.ylabel('Episode reward')
    plt.title(f'Phase {phase} — reward curves')
    plt.legend()
    plt.savefig(f'docs/plots/phase_{phase}_rewards.png', dpi=120, bbox_inches='tight')
    plt.show()